# CYR-GPU-014 / R1C — compatibility launcher v3

Pre-execution repair only: the original runner passed unsupported AdamW constructor keywords. This launcher binds the audited compatibility wrapper. Scientific arms, seeds, data, thresholds and horizons are unchanged. Select **T4 GPU** and **Run all**.


In [ ]:
import sys, json, hashlib, subprocess
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r1c')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='cymek-500m-readiness'
EXEC='b9e4689bdcfa24a3c7d50b2f337c4e702de0bb8a'
OLD_EXEC='2a71cea10ebb7a231834b6b112c49e268e9631a5'
if not REPO.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','220',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','220'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'],check=True)
live=json.loads((REPO/'docs/cymek/experiments/CYR-GPU-014-R1C/RUN_READINESS_V2.json').read_text())
assert live['status']=='READY_FOR_OPERATOR_COLAB_CUDA_PREEXECUTION_GATE' and live['scientific_result_status']=='NOT_EXECUTED'
assert live['frozen_compatibility_executable_commit']==EXEC
pre_text=(REPO/'docs/cymek/experiments/CYR-GPU-014-R1C/PREREGISTRATION.json').read_text()
PRE=Path('/content/CYR_GPU_014_R1C_PREREGISTRATION.json'); PRE.write_text(pre_text)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXEC],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==EXEC
expected={'anra_v5/cyr_gpu014_r1c_run_v2.py':'1078f0527f9e0c5ba6a94be6659209c0ab71a1cb','tests/test_v5_cyr_gpu014_r1c_compat.py':'7e82aec3b79ed27478efe48a36c3ca68bb962648','anra_v5/cyr_gpu014_r1c_run.py':'5bd880dea143e14348019b9c4f959c760a92f051'}
for p,sha in expected.items(): assert subprocess.check_output(['git','-C',str(REPO),'hash-object',p],text=True).strip()==sha,(p,sha)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','numpy'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu014_r1c_compat.py','tests/test_v5_cyr_gpu014_r1c.py','tests/test_v5_cyr_gpu013_r1b.py','tests/test_v5_cyr_gpu012_r1.py','-q'],cwd=REPO,check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-014-R1C'); OUT.mkdir(parents=True,exist_ok=True)
binding_path=OUT/'EXECUTABLE_BINDING.json'; gate=OUT/'PREEXECUTION_GATE.json'; campaign=OUT/'CAMPAIGN_RECEIPT.json'
new_binding={'scientific_executable_commit':EXEC,'original_scientific_executable_commit':OLD_EXEC,'compatibility_wrapper':'anra_v5/cyr_gpu014_r1c_run_v2.py','preregistration_raw_sha256':hashlib.sha256(pre_text.encode()).hexdigest()}
if binding_path.exists():
    old=json.loads(binding_path.read_text())
    if old!=new_binding:
        gate_pass=gate.exists() and json.loads(gate.read_text()).get('status')=='PASS'
        if gate_pass or campaign.exists(): raise RuntimeError('Refusing compatibility migration after scientific execution state exists')
        (OUT/'EXECUTABLE_BINDING_PRECOMPAT.json').write_text(json.dumps(old,indent=2)+'\n')
        binding_path.write_text(json.dumps(new_binding,indent=2)+'\n')
else: binding_path.write_text(json.dumps(new_binding,indent=2)+'\n')
if gate.exists() and json.loads(gate.read_text()).get('status')=='PASS': print('Existing R1C PREEXECUTION GATE: PASS — reuse')
else:
    cmd=[sys.executable,'-m','anra_v5.cyr_gpu014_r1c_run_v2','--mode','preflight','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE)]
    print('Running repaired R1C preflight:', ' '.join(cmd), flush=True)
    subprocess.run(cmd,cwd=REPO,check=True)
    assert json.loads(gate.read_text()).get('status')=='PASS'
print('R1C PREEXECUTION GATE: PASS')


In [ ]:
cmd=[sys.executable,'-m','anra_v5.cyr_gpu014_r1c_run_v2','--mode','run','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE)]
print('Starting/resuming frozen R1C campaign...',flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
from google.colab import files
campaign=json.loads((OUT/'CAMPAIGN_RECEIPT.json').read_text()); status=campaign.get('status')
print('STATUS:',status,'COMPLETED:',campaign.get('completed_arm_count'),'/24','VERDICT:',campaign.get('decision',{}).get('verdict'))
bundle=OUT/('CYMEK_R1C_SOFTMAX_MECHANISM_RESULTS.zip' if status=='COMPLETE' else 'CYMEK_R1C_SOFTMAX_MECHANISM_PARTIAL.zip')
actual=hashlib.sha256(bundle.read_bytes()).hexdigest(); print('BUNDLE:',bundle.name,'SHA256:',actual)
files.download(str(bundle))
